# high-dimensional regime — Python demo

Numerical companion to the entry [high-dimensional regime](https://dictionaryofml.org/terms/highdimregime.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'highdimregime' (high-dimensional regime). A linear model predicts the air temperature at 15:00 (UTC) of a day at Krems (GeoSphere station 3805) from the 15:00 readings of the previous d days. A day qualifies only when all 60 of its predecessors are in the record, so the two days the hourly record drops, 2023-12-31 and 2024-01-12, take the first weeks of 2024 out with them. The training set holds m = 30 days of 2023, the validation set the qualifying days of 2024. The feature count d runs from 1 to 60, so it crosses the sample size m. For d < m the hypothesis learned by least squares has a training error of the same order as its validation error. At d = m the feature matrix has rank m, the learned hypothesis interpolates the training set, the training error is zero, and the validation error is 65 times the validation error at d = 1: the training error has stopped saying anything about the loss on days outside the training set. Two routes restore a usable method on the same features, and the demo recomputes both: a penalty term (ridge regression with a fixed regularization parameter) and dimensionality reduction (least squares on the first five principal directions of the training features).

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/highdimregime.py`](https://dictionaryofml.org/terms/highdimregime.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 highdimregime.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "highdimregime.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""The high-dimensional regime at Krems an der Donau: thirty training days
and up to sixty features, with the training error collapsing to zero and the
validation error exploding as the feature count reaches the sample size.

Purpose
-------
Numerical companion to the glossary entry 'highdimregime' (high-dimensional
regime).  A linear model predicts the air temperature at 15:00 (UTC) of a
day at Krems (GeoSphere station 3805) from the 15:00 readings of the
previous d days.  A day qualifies only when all 60 of its predecessors are
in the record, so the two days the hourly record drops, 2023-12-31 and
2024-01-12, take the first weeks of 2024 out with them.  The training set
holds m = 30 days of 2023, the validation set the qualifying days of 2024.  The feature count d runs from 1 to
60, so it crosses the sample size m.  For d < m the hypothesis learned by
least squares has a training error of the same order as its validation
error.  At d = m the feature matrix has rank m, the learned hypothesis
interpolates the training set, the training error is zero, and the
validation error is 65 times the validation error at d = 1: the training
error has stopped saying anything about the loss on days outside the
training set.  Two routes restore a usable method on the
same features, and the demo recomputes both: a penalty term (ridge
regression with a fixed regularization parameter) and dimensionality
reduction (least squares on the first five principal directions of the
training features).

Self-contained (numpy and matplotlib only), fixed seeds; the GeoSphere
download falls back to the committed CSV.

Blocks
------
[B-data]     The 15:00 (UTC) air temperature at Krems for 2023 and 2024
             (729 of the 731 days carry it).
[B-split]    The m = 30 training days, drawn from the 2023 days that have
             60 consecutive earlier days on record, and the qualifying
             validation days of 2024.
[B-leastsq]  Training and validation error of the least-squares hypothesis
             for every feature count d = 1, ..., 60.
[B-interp]   At d = m the rank of the feature matrix is m, the training
             error is zero, and the minimum-norm solution has a large
             Euclidean norm.
[B-restore]  A penalty term and dimensionality reduction keep the
             validation error below 33 squared degrees at every d.

Outputs
-------
highdimregime_weather.csv  : date, t15 for 2023 and 2024 (committed copy).
highdimregime_errors.csv   : d, training error, the same clipped at 1e-3
                             for a logarithmic axis, validation error of
                             least squares.
highdimregime_restored.csv : d, validation error of least squares, of ridge
                             regression, and after dimensionality reduction.
highdimregime.png          : matplotlib preview (checking only).
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import datetime
import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

DMAX = 60          # largest feature count
SAMPLESIZE = 30    # m, the number of training days
REGPARAM = 100.0   # alpha, the regularization parameter of ridge regression
KEPTDIM = 5        # d', the number of kept principal directions

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-data]** The 15:00 (UTC) air temperature at Krems for 2023 and 2024 (729 of the 731 days carry it).

In [ ]:
print("[B-data] the 15:00 (UTC) air temperature at Krems, 2023 and 2024")
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1h?parameters=tl&station_ids=3805"
       "&start=2023-01-01T00:00&end=2024-12-31T23:00")
CSV = OUT_DIR / "highdimregime_weather.csv"
try:
    with urllib.request.urlopen(URL, timeout=300) as resp:
        payload = json.load(resp)
    readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
    byday = {}
    for stamp, value in zip(payload["timestamps"], readings):
        if value is None:                  # the archive drops whole days
            continue
        if stamp[11:16] == "15:00":        # the stamps are UTC, and so is
            byday[stamp[:10]] = float(value)   # the time, so no conversion
    records = sorted(byday.items())
    with open(CSV, "w") as fh:
        fh.write("date,t15\n")
        for day, hi in records:
            fh.write(f"{day},{hi}\n")
    print(f"    downloaded {len(records)} days with a 15:00 reading")
except OSError as err:
    print(f"    download failed ({err}); reading the committed CSV")
    rows = CSV.read_text().strip().splitlines()[1:]
    records = [(d, float(a)) for d, a in (r.split(",") for r in rows)]

check("[B-data] 729 of the 731 days of 2023 and 2024 carry a 15:00 reading "
      "(the hourly record drops 2023-12-31 and 2024-01-12)",
      len(records) == 729)
check("[B-data] no kept day is missing its 15:00 reading",
      all(hi is not None for _, hi in records))
t15 = np.array([hi for _, hi in records], dtype=float)
dates = [d for d, _ in records]
day23 = np.array([d.startswith("2023") for d in dates])
check(f"[B-data] the readings range from {t15.min():.1f} to {t15.max():.1f} "
      f"degrees Celsius", -20.0 < t15.min() < t15.max() < 45.0)

**[B-split]** The m = 30 training days, drawn from the 2023 days that have 60 consecutive earlier days on record, and the qualifying validation days of 2024.

In [ ]:
print("[B-split] the training days of 2023 and the validation days of 2024")
# lag k must be the kth earlier CALENDAR day, so a day qualifies only when
# its DMAX predecessors are all in the record
ordinal = np.array([datetime.date.fromisoformat(d).toordinal()
                    for d in dates])
qualifies = np.zeros(len(dates), dtype=bool)
qualifies[DMAX:] = ordinal[DMAX:] - ordinal[:-DMAX] == DMAX
candidates = np.flatnonzero(qualifies & day23)
rng = np.random.default_rng(2026)
train_idx = np.sort(rng.choice(candidates, size=SAMPLESIZE, replace=False))
val_idx = np.flatnonzero(qualifies & ~day23)
check(f"[B-split] {len(train_idx)} training days of 2023, the first "
      f"{dates[train_idx[0]]} and the last {dates[train_idx[-1]]}",
      len(train_idx) == SAMPLESIZE and len(set(train_idx.tolist())) == SAMPLESIZE)
check(f"[B-split] {len(val_idx)} validation days of 2024, from "
      f"{dates[val_idx[0]]} on: the days before it reach back over "
      f"2023-12-31 or 2024-01-12, which the record lacks",
      len(val_idx) == 294 and dates[val_idx[0]] == "2024-03-13")
check("[B-split] no day belongs to both sets",
      not set(train_idx.tolist()) & set(val_idx.tolist()))
check(f"[B-split] every day of either set has {DMAX} consecutive earlier "
      f"days on record",
      bool(np.all(qualifies[train_idx]) and np.all(qualifies[val_idx])))


def features(idx, nr_features):
    """The 15:00 readings of the nr_features days before each day in idx."""
    return np.column_stack([t15[idx - lag]
                            for lag in range(1, nr_features + 1)])


def mean_sqerr(idx, weights, nr_features):
    residual = t15[idx] - features(idx, nr_features) @ weights
    return float(np.mean(residual ** 2))

**[B-leastsq]** Training and validation error of the least-squares hypothesis for every feature count d = 1, ..., 60.

In [ ]:
print("[B-leastsq] least squares for every feature count d = 1, ..., 60")
dims = np.arange(1, DMAX + 1)
trainerr = np.empty(DMAX)
valerr = np.empty(DMAX)
solnorm = np.empty(DMAX)
ranks = np.empty(DMAX, dtype=int)
for i, d in enumerate(dims):
    Xtrain = features(train_idx, d)
    what = np.linalg.lstsq(Xtrain, t15[train_idx], rcond=None)[0]
    trainerr[i] = mean_sqerr(train_idx, what, d)
    valerr[i] = mean_sqerr(val_idx, what, d)
    solnorm[i] = float(np.linalg.norm(what))
    ranks[i] = int(np.linalg.matrix_rank(Xtrain))

check(f"[B-leastsq] d = 1 (yesterday alone): training error "
      f"{trainerr[0]:.2f}, validation error {valerr[0]:.2f} squared "
      f"degrees, the two of the same order",
      0.5 < valerr[0] / trainerr[0] < 2.0)
check(f"[B-leastsq] the training error decreases with d, from "
      f"{trainerr[0]:.2f} at d = 1 to {trainerr[SAMPLESIZE - 2]:.2f} at "
      f"d = {SAMPLESIZE - 1}", np.all(np.diff(trainerr[:SAMPLESIZE]) < 1e-9))
check(f"[B-leastsq] d = {SAMPLESIZE - 1}: training error "
      f"{trainerr[SAMPLESIZE - 2]:.2f}, validation error "
      f"{valerr[SAMPLESIZE - 2]:.1f} squared degrees",
      valerr[SAMPLESIZE - 2] > 20 * trainerr[SAMPLESIZE - 2])
check(f"[B-leastsq] the validation error is smallest at d = "
      f"{dims[valerr.argmin()]} with {valerr.min():.2f} squared degrees",
      dims[valerr.argmin()] < SAMPLESIZE)
check(f"[B-leastsq] the validation error is largest at d = "
      f"{dims[valerr.argmax()]} with {valerr.max():.0f} squared degrees, "
      f"just past the sample size m = {SAMPLESIZE}",
      SAMPLESIZE <= dims[valerr.argmax()] <= SAMPLESIZE + 2)

**[B-interp]** At d = m the rank of the feature matrix is m, the training error is zero, and the minimum-norm solution has a large Euclidean norm.

In [ ]:
print("[B-interp] interpolation of the training set at d = m")
at_m = SAMPLESIZE - 1
check(f"[B-interp] the rank of the feature matrix is min(d, m): "
      f"{ranks[0]} at d = 1, {ranks[at_m]} at d = {SAMPLESIZE}, "
      f"{ranks[-1]} at d = {DMAX}",
      np.all(ranks == np.minimum(dims, SAMPLESIZE)))
check(f"[B-interp] at d = m = {SAMPLESIZE} the training error is "
      f"{trainerr[at_m]:.2e}, zero up to rounding, so the learned "
      f"hypothesis interpolates the training set", trainerr[at_m] < 1e-20)
check(f"[B-interp] every d from {SAMPLESIZE} to {DMAX} interpolates the "
      f"training set", np.all(trainerr[at_m:] < 1e-18))
check(f"[B-interp] at d = m the validation error is {valerr[at_m]:.0f} "
      f"squared degrees, {valerr[at_m] / valerr[0]:.0f} times the "
      f"validation error {valerr[0]:.2f} at d = 1",
      valerr[at_m] > 50 * valerr[0])
check(f"[B-interp] the minimum-norm solution grows from "
      f"{solnorm[0]:.2f} at d = 1 to {solnorm[at_m]:.1f} at d = m",
      solnorm[at_m] > 10 * solnorm[0])

**[B-restore]** A penalty term and dimensionality reduction keep the validation error below 33 squared degrees at every d.

In [ ]:
print("[B-restore] a penalty term and dimensionality reduction")
ridgeerr = np.empty(DMAX)
pcaerr = np.empty(DMAX)
for i, d in enumerate(dims):
    Xtrain = features(train_idx, d)
    ytrain = t15[train_idx]
    ridge = np.linalg.solve(Xtrain.T @ Xtrain + REGPARAM * np.eye(d),
                            Xtrain.T @ ytrain)
    ridgeerr[i] = mean_sqerr(val_idx, ridge, d)
    kept = min(KEPTDIM, d)
    basis = np.linalg.svd(Xtrain, full_matrices=False)[2][:kept].T
    coeff = np.linalg.lstsq(Xtrain @ basis, ytrain, rcond=None)[0]
    pcaerr[i] = mean_sqerr(val_idx, basis @ coeff, d)

check(f"[B-restore] ridge regression with alpha = {REGPARAM:.0f} keeps the "
      f"validation error between {ridgeerr.min():.1f} and "
      f"{ridgeerr.max():.1f} squared degrees over all 60 feature counts",
      ridgeerr.max() < 35.0)
check(f"[B-restore] least squares on the first {KEPTDIM} principal "
      f"directions keeps it between {pcaerr.min():.1f} and "
      f"{pcaerr.max():.1f}", pcaerr.max() < 25.0)
check(f"[B-restore] at d = m both stay near their value at d = 1: ridge "
      f"{ridgeerr[at_m]:.1f} against {ridgeerr[0]:.1f}, reduced "
      f"{pcaerr[at_m]:.1f} against {pcaerr[0]:.1f}",
      ridgeerr[at_m] < 3 * ridgeerr[0] and pcaerr[at_m] < 3 * pcaerr[0])
check(f"[B-restore] at d = m both are below the least-squares validation "
      f"error {valerr[at_m]:.0f} by a factor above 40",
      valerr[at_m] / max(ridgeerr[at_m], pcaerr[at_m]) > 40.0)
Xm = features(train_idx, SAMPLESIZE)
ridge_m = np.linalg.solve(Xm.T @ Xm + REGPARAM * np.eye(SAMPLESIZE),
                          Xm.T @ t15[train_idx])
basis_m = np.linalg.svd(Xm, full_matrices=False)[2][:KEPTDIM].T
coeff_m = np.linalg.lstsq(Xm @ basis_m, t15[train_idx], rcond=None)[0]
check(f"[B-restore] at d = m neither route interpolates the training set: "
      f"training error {mean_sqerr(train_idx, ridge_m, SAMPLESIZE):.2f} with "
      f"the penalty term and "
      f"{mean_sqerr(train_idx, basis_m @ coeff_m, SAMPLESIZE):.2f} after the "
      f"reduction, against {trainerr[at_m]:.2e} for least squares",
      min(mean_sqerr(train_idx, ridge_m, SAMPLESIZE),
          mean_sqerr(train_idx, basis_m @ coeff_m, SAMPLESIZE)) > 1.0)


# ---- CSVs and preview
FLOOR = 1e-3   # a log axis cannot draw a zero training error
with open(OUT_DIR / "highdimregime_errors.csv", "w") as fh:
    fh.write("d,trainerr,trainplot,valerr\n")
    for i, d in enumerate(dims):
        fh.write(f"{d},{trainerr[i]:.6g},{max(trainerr[i], FLOOR):.6g},"
                 f"{valerr[i]:.6g}\n")
with open(OUT_DIR / "highdimregime_restored.csv", "w") as fh:
    fh.write("d,leastsquares,ridge,reduced\n")
    for i, d in enumerate(dims):
        fh.write(f"{d},{valerr[i]:.6g},{ridgeerr[i]:.6g},{pcaerr[i]:.6g}\n")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.9))
ax1.semilogy(dims, np.maximum(trainerr, FLOOR), "k-", lw=1.1,
             label=f"training error (at {FLOOR:g} where it is zero)")
ax1.semilogy(dims, valerr, "k--", lw=1.1, label="validation error")
ax1.axvline(SAMPLESIZE, color="black", ls=":", lw=0.9,
            label=f"d = m = {SAMPLESIZE}")
ax1.set_xlabel("number d of features")
ax1.set_ylabel("average squared error loss")
ax1.set_title("the gap opens as d reaches m")
ax1.legend(frameon=False, fontsize=8)
ax2.semilogy(dims, valerr, "k--", lw=1.1, label="least squares")
ax2.semilogy(dims, ridgeerr, "k-", lw=1.1,
             label=f"penalty term, alpha = {REGPARAM:.0f}")
ax2.semilogy(dims, pcaerr, "k:", lw=1.3,
             label=f"reduced to {KEPTDIM} directions")
ax2.set_xlabel("number d of features")
ax2.set_ylabel("validation error")
ax2.set_title("two routes keep the method usable")
ax2.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "highdimregime.png", dpi=110)

n_ok = sum(ok for _, ok in report)
print(f"\n{n_ok}/{len(report)} checks pass")
if n_ok != len(report):
    raise SystemExit(1)